In [1]:
import sys
print(sys.executable)
import os
from scipy.stats import gaussian_kde

import numpy as np
from scipy.special import erf, erfinv
#from math import erf
from numpy.linalg import norm
from mlmm import VelocityData, plot_learn_curve, n_fold_cv, fileinfo,VelocityDataOmegaData, OmegaData

import matplotlib.pyplot as plt

from datetime import date
today = date.today().strftime("%b-%d-%Y")
import periodictable

import sys
from mlmm import func_postprocess
import time
import pandas as pd

from scipy.stats import norm
from scipy.stats import gaussian_kde

from sklearn.mixture import BayesianGaussianMixture
from sklearn.mixture import GaussianMixture
import joblib

import seaborn as sns
from scipy.stats import gaussian_kde

c:\Users\jespe\miniconda3\python.exe


In [5]:
def prepare_data_2(path_to_data, path_save_data, system_6D, system_10D, system_omega, wall_temp, Liao_Transfer_Function):
    x_data = []
    y_data = []

    x_omega = []
    y_omega = []

    x_MD = []
    y_MD = []

    for file in os.listdir(path_to_data):

        if file.endswith(".txt") and "MD" in file:
            x_MD.append(os.path.join(path_to_data,file))
            y_MD.append('')

        if file.endswith(".txt") and "omega" in file:
            x_omega.append(os.path.join(path_to_data,file))
            y_omega.append('')
    print("Data files found:",x_MD)


    ######################################################################################
    #------------------------------- Getting translational velocity data
    ####--------------------------------------------------------------------------------------------------
    for x_data_file,y_data_file in zip(x_MD,y_MD):
        

        conf = VelocityData(x_data_file, frames=None)#[0,30000])
        conf.getRep(rep='vxvyvz',nuc=None) #rep options: vxvyvz, vel2norm, vx2,vy2,vz2,vx,vy,vz
        file_name=x_data_file.replace('.','/')
        file_name=file_name.split('/')
        set_name = file_name[1]
        X = conf.X
        y = conf.y
        if "He" in path_to_data:
            mass = getattr(periodictable,'He').mass
            gas_name = 'He'
        elif "Ar" in path_to_data:
            mass = getattr(periodictable,'Ar').mass
            gas_name = 'Ar'
        elif "H2" in path_to_data:
            mass =2* getattr(periodictable,'H').mass
            gas_name = 'H2'
            l_b=0.741e-10
            mass_kg = mass * 0.001 / av_num
            I=(mass_kg/4)*l_b**2
        elif "N2" in path_to_data:
            mass =2* getattr(periodictable,'N').mass
            gas_name = 'N2'
            l_b=1.097e-10
            mass_kg = mass * 0.001 / av_num
            I=(mass_kg/4)*l_b**2
        else:
            print ("Unable to identify impinging atom type...")
            ele = str(input("Enter the impinging atom symbol : "))
            mass = getattr(periodictable,ele).mass
            v_mp = np.sqrt(2 * kB * wall_temp / mass_kg)
            omega_mp = np.sqrt(2 * kB * wall_temp / I)
        print ('Atomic mass for {} molecule is {:1.4f} \n'.format(gas_name,mass))
        n_MD = X.shape[0]
        print('Number of MD tr velocity data points: {} \n'.format(n_MD))
        # Implementing Liao Transfer function on perpendicular velocity component
        v_TF,T_in,T_out,theta_in,theta_out=func_postprocess.liao_transform(X,mass,'y')
        v_RTF=func_postprocess.liao_R_transform(v_TF,theta_in,theta_out,'y')
        MAC_x,MAC_y,MAC_z,EAC_x,EAC_y,EAC_z,EAC_tr=func_postprocess.compute_AC_correlation_method(X,'Ar','y') #
        AC_MD_tr = [MAC_x,MAC_y,MAC_z,EAC_x,EAC_y,EAC_z,EAC_tr]


    ######################################################################################
    #------------------------------- Getting angular velocity data
    ####--------------------------------------------------------------------------------------------------
    if system_omega:
        for x_data_file2,y_data_file2 in zip(x_omega,y_omega):
            #conf2 = OmegaData(x_data_file2, frames=frames)#[0,30000])
            conf2 = OmegaData(x_data_file2, frames=None)#[0,30000])
            conf2.getRep(rep='omega1omega2',nuc=None) #rep options: vxvyvz, vel2norm, vx2,vy2,vz2,vx,vy,vz
            omega_file_name = x_data_file2.replace('.','/')
            omega_file_name = omega_file_name.split('/')
            omega_name = omega_file_name[1]
            print(omega_name)
            X2 = conf2.X2
            y2 = conf2.y2
            print('Number of MD rot velocity data points: {} \n'.format(X2.shape[0]))
            X2_TF=np.copy(X2)
            Y2_TF=np.copy(y2)
            omega_TF=np.vstack((X2_TF,-X2_TF))
            y2_TF=np.hstack((Y2_TF,-Y2_TF))   

    ######################################################################################
    #------------------------------- Implementing GM model on 10D data
    ####--------------------------------------------------------------------------------------------------                   

    if system_10D:
        SPEED_THRESHOLD = 45  # m/s
        vel_tr = np.copy(X)

        # Speed filtering
        def compute_speed(X: np.ndarray) -> np.ndarray:
            """Compute the speed from the 3D velocity components."""
            return np.sqrt(X[:, 0]**2 + X[:, 1]**2 + X[:, 2]**2)
        speed_in = compute_speed(vel_tr)
        vel_tr = vel_tr[speed_in > SPEED_THRESHOLD]
        X2 = X2[speed_in > SPEED_THRESHOLD]

        vel_tr[:,1]=np.abs(vel_tr[:,1])
        vel_tr[:,4]=np.abs(vel_tr[:,4])
        data_for_AC_MD = np.concatenate((vel_tr,X2),axis=1)

        
        if Liao_Transfer_Function:
            #omega_bond = np.copy(X2)
            v_tr=np.copy(v_TF[:n_MD,:])[speed_in > SPEED_THRESHOLD]

            v_omega = np.copy(X2)
            v_mp = np.sqrt(2 * kB * wall_temp / mass_kg)
            omega_mp = np.sqrt(2 * kB * wall_temp / I)


            #--- Normalizing the translational and rotational velocities
            data_train_10D = np.concatenate((v_tr*conv_v/v_mp,X2*conv_omega/omega_mp),axis=1)

            print('The total number of training (after speed filtering) points is: {} \n'.format(data_train_10D.shape[0]))

        else:
            data_train_10D = data_for_AC_MD
            print('The total number of training points is: {} \n'.format(data_train_10D.shape[0]))
    
    X_df = pd.DataFrame(data_train_10D, columns=['vx_in', 'vy_in', 'vz_in', 'vx_out', 'vy_out', 'vz_out', 'omega_1_in', 'omega_2_in', 'omega_1_out', 'omega_2_out'], copy=True)

    
    return X_df

In [6]:
# N_G as from the literature. TODO: plot N_G vs accomodation coefficient, as is done in the literature
N_G = 600

# Create a function to make and fit a GMM model
def initialize_gmm(X, n_components):
    gmm = GaussianMixture(n_components=n_components, covariance_type='full', tol=0.001)
    gmm.fit(X)
    return gmm

# Create a function to make and fit a Bayesian GMM model
def initialize_bayesian_gmm(X, n_components):
    bayesian_gmm = BayesianGaussianMixture(n_components=n_components)
    bayesian_gmm.fit(X)
    return bayesian_gmm

def save_gmm_model(model: GaussianMixture | BayesianGaussianMixture, filename: str):
    os.makedirs("trained_models", exist_ok=True)  # Ensure the directory exists
    
    with open("trained_models/" + filename, "wb") as f:
        joblib.dump(model, f, protocol=5)

def load_gmm_model(filename: str) -> GaussianMixture | BayesianGaussianMixture:
    with open("trained_models/" + filename, "rb") as f:
        model = joblib.load(f)
    return model


# TODO: lookup how the accomodation coefficient is calculated in the paper and create a for loop to calculate it for all possible models (N_G =1 to 1000)


## Functions for training, saving and loading both models

In [7]:
# N_G as from the literature. TODO: plot N_G vs accomodation coefficient, as is done in the literature
N_G = 600

# Create a function to make and fit a GMM model
def initialize_gmm(X, n_components):
    gmm = GaussianMixture(n_components=n_components, covariance_type='full', tol=0.001)
    gmm.fit(X)
    return gmm

# Create a function to make and fit a Bayesian GMM model
def initialize_bayesian_gmm(X, n_components):
    bayesian_gmm = BayesianGaussianMixture(n_components=n_components)
    bayesian_gmm.fit(X)
    return bayesian_gmm

def save_gmm_model(model: GaussianMixture | BayesianGaussianMixture, filename: str):
    os.makedirs("trained_models", exist_ok=True)  # Ensure the directory exists
    
    with open("trained_models/" + filename, "wb") as f:
        joblib.dump(model, f, protocol=5)

def load_gmm_model(filename: str) -> GaussianMixture | BayesianGaussianMixture:
    with open("trained_models/" + filename, "rb") as f:
        model = joblib.load(f)
    return model


# TODO: lookup how the accomodation coefficient is calculated in the paper and create a for loop to calculate it for all possible models (N_G =1 to 1000)


In [8]:
def visualize_pdf_10D(X_df, gmm_model, bayesian_model, title, N_G_gmm, N_G_bayesian):
    """
    Function: visualize_pdf
    Visualizes the probability density function (PDF) of the features in the dataset using both GMM and Bayesian GMM models.
    @param X_df: DataFrame containing the features.
    @param gmm_model: Fitted Gaussian Mixture Model.
    @param bayesian_model: Fitted Bayesian Gaussian Mixture Model.
    @param title (string): Title for the plot.
    """
    
    # Extract marginal pdf for each feature from the GMM model
    fig, ax = plt.subplots(1, 5, figsize=(10, 5))

    labels = ["vx_out", "vy_out", "vz_out", "Omega_1_out", "Omega_2_out"]
    feature_index = [3, 4, 5, 8, 9]
    xlim = np.linspace(-5, 5, 1000)

    for i in range(5):
        pdf_gmm = np.zeros_like(xlim)
        pdf_bayesian = np.zeros_like(xlim)

        for j in range(N_G_gmm):
            # Extract the mean, covariance, and weight for each gmm component
            mean_gmm = gmm_model.means_[j, feature_index[i]]
            cov_gmm = gmm_model.covariances_[j, feature_index[i], feature_index[i]]
            weight_gmm = gmm_model.weights_[j]
            pdf_gmm += weight_gmm * norm.pdf(xlim, loc=mean_gmm, scale=np.sqrt(cov_gmm))
        
        for j in range(N_G_bayesian):   
            # Extract the mean, covariance, and weight for each bayesian component
            mean_bayesian = bayesian_model.means_[j, feature_index[i]]
            cov_bayesian = bayesian_model.covariances_[j, feature_index[i], feature_index[i]]
            weight_bayesian = bayesian_model.weights_[j]
            pdf_bayesian += weight_bayesian * norm.pdf(xlim, loc=mean_bayesian, scale=np.sqrt(cov_bayesian))

            # Compute histogram
            

        # Visualize the pdf of the gmm as a line plot
        ax[i].plot(xlim, pdf_gmm, c='red', alpha=0.8, label='GMM PDF')
        
        # Compute KDE of the actual data
        data_kde = gaussian_kde(X_df.iloc[:, feature_index[i]])
        pdf_data = data_kde(xlim)

        # Plot the data KDE as the empirical PDF
        ax[i].plot(xlim, pdf_data, c='blue', alpha=0.6, label='MD PDF')

        
        # Visualize the pdf of the bayesian model as a line plot
        ax[i].plot(xlim, pdf_bayesian, c='green', alpha=0.8, label='Bayesian PDF')
        ax[i].set_ylim([0, 0.7])
        ax[i].set_xlabel(f'{labels[i]}')
        ax[i].set_title(f'PDF of {labels[i]}')
        
        
    ax[0].set_ylabel("PDF")
    ax[2].legend(loc=1)
    fig.tight_layout()
    plt.show()
    fig.suptitle("pdf " + title, fontsize=16)
    fig.savefig("pdf_" + title, dpi=300, bbox_inches='tight')


In [9]:
def heatmap_10D(X_df, gmm_model, bayesian_model, title, n_samples=None):
    """
    Function: heatmap_10D
    Visualizes the heatmap of the features in the dataset using both GMM and Bayesian GMM models.
    @param X_df: DataFrame containing the features.
    @param gmm_model: Fitted Gaussian Mixture Model.
    @param bayesian_model: Fitted Bayesian Gaussian Mixture Model.
    @param title (string): Title for the plot.
    """
    if n_samples is None:
        n_samples = X_df.shape[0] # Use the number of rows in X_df if n_samples is not provided

    fntsz = 20
    fig, ax = plt.subplots(5, 3, figsize=(30, 30))
    labels = ["vx_out", "vy_out", "vz_out", "Omega_1_out", "Omega_2_out"]

    # generate samples from the GMM and Bayesian GMM models

    samples_gmm, _ = gmm_model.sample(n_samples)
    samples_bayesian, _ = bayesian_model.sample(n_samples)
    samples_gmm_df = pd.DataFrame(samples_gmm, columns=['vx_in', 'vy_in', 'vz_in', 'vx_out', 'vy_out', 'vz_out', 'omega_1_in', 'omega_2_in', 'omega_1_out', 'omega_2_out'], copy=True)
    samples_bayesian_df = pd.DataFrame(samples_bayesian, columns=['vx_in', 'vy_in', 'vz_in', 'vx_out', 'vy_out', 'vz_out', 'omega_1_in', 'omega_2_in', 'omega_1_out', 'omega_2_out'], copy=True)

    in_index = [0, 1, 2, 6, 7]  # Indices for the input features
    out_index = [3, 4, 5, 8, 9]  # Indices for the output features

    # Loop through the features to create subplots
    for r in range(5):
        # Compute density for MD data
        xy_MD = np.vstack([X_df.iloc[:n_samples, in_index[r]], X_df.iloc[:n_samples, out_index[r]]])
        density_MD = gaussian_kde(xy_MD)(xy_MD)

        # plot the MD data
        ax[r, 0].scatter(X_df.iloc[:n_samples, in_index[r]], X_df.iloc[:n_samples, out_index[r]], c=density_MD, s=15, cmap='jet', alpha=0.6)

        # Compute density for GMM samples
        xy_GMM = np.vstack([samples_gmm_df.iloc[:n_samples, in_index[r]], samples_gmm_df.iloc[:n_samples, out_index[r]]])
        density_GMM = gaussian_kde(xy_GMM)(xy_GMM)

        # plot the GMM data
        ax[r, 1].scatter(samples_gmm_df.iloc[:n_samples, in_index[r]], samples_gmm_df.iloc[:n_samples, out_index[r]], c=density_GMM, s=15, cmap='jet', alpha=0.6)

        # Compute density for Bayesian samples
        xy_Bayesian = np.vstack([samples_bayesian_df.iloc[:n_samples, in_index[r]], samples_bayesian_df.iloc[:n_samples, out_index[r]]])
        density_Bayesian = gaussian_kde(xy_Bayesian)(xy_Bayesian)

        # plot the Bayesian data
        ax[r, 2].scatter(samples_bayesian_df.iloc[:n_samples, in_index[r]], samples_bayesian_df.iloc[:n_samples, out_index[r]], c=density_Bayesian, s=15, cmap='jet', alpha=0.6)

    # row labels
    ax[0, 0].set_ylabel('vx', fontsize=fntsz)
    ax[1, 0].set_ylabel('vy', fontsize=fntsz)
    ax[2, 0].set_ylabel('vz', fontsize=fntsz)
    ax[3, 0].set_ylabel('Omega_1', fontsize=fntsz)
    ax[4, 0].set_ylabel('Omega_2', fontsize=fntsz)
    # column labels
    ax[0, 0].set_title("MD", fontsize=fntsz)
    ax[0, 1].set_title("GMM", fontsize=fntsz)
    ax[0, 2].set_title("Bayesian GMM", fontsize=fntsz)

    fig.suptitle("heatmap "+ title, fontsize=24)
    fig.savefig("heatmap_"+ title + '.png', dpi=300, bbox_inches='tight')

# Prepare the DF's

In [10]:
er_type = 'MAE'

system_6D = False 
system_10D = True # Assignment C uses 10D data since we now take rotational velocity data into account
system_omega = True 
Liao_Transfer_Function = False # we use a different liao transform function

kB,conv_v,conv_omega,av_num = 1.38064852e-23,1.0e2,1.0e12,6.022e23

In [49]:
# prepare the X_df's.

"""
H2_omega_Sw_20_B300_Th_300.txt and H2_Vel_MD_Sw_20_B300_Th_300.txt
Bottom wall temperature = 300 K
Top wall temperature = 300 K
"""
path_to_data = 'Data_H2_assignment_bayesian_C_1'
path_save_data='./'+path_to_data
wall_temp = 300 # Wall temperature in Kelvin
X_df_1_C = prepare_data_2(path_to_data, path_save_data, system_6D, system_10D, system_omega, wall_temp, Liao_Transfer_Function)[:3000]
X_df_C = X_df_1_C # For if I want to use the first dataframe only, for testing purposes.
X_df_C.head()


"""
H2_omega_Sw_20_B300_Th_500.txt and H2_Vel_MD_Sw_20_B300_Th_500.txt
Bottom wall temperature = 300 K
Top wall temperature = 500 K
"""
path_to_data = 'Data_H2_assignment_bayesian_C_3'
path_save_data='./'+path_to_data
wall_temp = 300 # Wall temperature in Kelvin
X_df_2_C = prepare_data_2(path_to_data, path_save_data, system_6D, system_10D, system_omega, wall_temp, Liao_Transfer_Function)[:3000]


"""
H2_omega_Sw_20_T500_Tb_300.txt and H2_Vel_MD_Sw_20_T500_Tb_300.txt
Bottom wall temperature = 300 K
Top wall temperature = 500 K
"""
path_to_data = 'Data_H2_assignment_bayesian_C_3'
path_save_data='./'+path_to_data
wall_temp = 300 # Wall temperature in Kelvin
X_df_3_C = prepare_data_2(path_to_data, path_save_data, system_6D, system_10D, system_omega, wall_temp, Liao_Transfer_Function)[:3000]

Data files found: ['Data_H2_assignment_bayesian_C_1\\H2_Vel_MD_Sw_20_B300_Th_300.txt']
Generating Representation
Not a float


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:01


Atomic mass for H2 molecule is 2.0159 

Number of MD tr velocity data points: 263301 

Generating Representation


0% [####                          ] 100% | ETA: 00:00:01

Not a float


...

Atomic mass for H2 molecule is 2.0159 

Number of MD tr velocity data points: 184022 

Generating Representation


0% [#########                     ] 100% | ETA: 00:00:00

Not a float


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


txt
Number of MD rot velocity data points: 184022 

The total number of training (after speed filtering) points is: 5729 



# Training and visualizing for Sw_20_B300_Th_300

In [12]:
# Train the gaussian mixture model and bayesian gaussian mixture model on the following data sets:

title1_C = "Sw_20_B300_Th_300"
N_G_gmm = 600  # Number of components for GMM
N_G_bayesian = 400  # Number of components for Bayesian GMM

gmm_model_1_C = initialize_gmm(X_df_1_C, N_G_gmm)
save_gmm_model(gmm_model_1_C, f'gmm_model_{title1_C}.pkl')
bayesian_model_1_C = initialize_bayesian_gmm(X_df_1_C, N_G_bayesian)
save_gmm_model(bayesian_model_1_C, f'bayesian_model_{title1_C}.pkl')

c:\Users\jespe\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=9.
  warnings.warn(
c:\Users\jespe\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=9.
  warnings.warn(


In [13]:
title1_C = "Sw_20_B300_Th_300"
N_G_gmm = 600  # Number of components for GMM
N_G_bayesian = 400  # Number of components for Bayesian GMM

gmm_model_1_C = load_gmm_model(f'gmm_model_{title1_C}.pkl')
bayesian_model_1_C = load_gmm_model(f'bayesian_model_{title1_C}.pkl')

# Seperate the visualization and training part
visualize_pdf_10D(X_df_1_C, gmm_model_1_C, bayesian_model_1_C, title1_C, N_G_gmm, N_G_bayesian)
# Plot heatmap of the GMM
heatmap_10D(X_df_1_C, gmm_model_1_C, bayesian_model_1_C, title1_C, n_samples=X_df_1_C.shape[0])

C:\Users\jespe\AppData\Local\Temp\ipykernel_34360\2729003733.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


KeyboardInterrupt: 

# Training and visualizing for Sw_20_B300_Th_500

In [ ]:
# Train the gaussian mixture model and bayesian gaussian mixture model on the following data sets:

title2_C = "Sw_20_B300_Th_500"
N_G_gmm = 600  # Number of components for GMM
N_G_bayesian = 400  # Number of components for Bayesian GMM

gmm_model_2_C = initialize_gmm(X_df_2_C, N_G_gmm)
save_gmm_model(gmm_model_2_C, f'gmm_model_{title2_C}.pkl')
bayesian_model_2_C = initialize_bayesian_gmm(X_df_2_C, N_G_bayesian)
save_gmm_model(bayesian_model_2_C, f'bayesian_model_{title2_C}.pkl')

In [ ]:
title2_C = "Sw_20_B300_Th_500"
gmm_model_2_C = load_gmm_model(f'gmm_model_{title2_C}.pkl')
bayesian_model_2_C = load_gmm_model(f'bayesian_model_{title2_C}.pkl')

# Seperate the visualization and training part
visualize_pdf_10D(X_df_2_C, gmm_model_2_C, bayesian_model_2_C, title2_C, N_G_gmm, N_G_bayesian)
# Plot heatmap of the GMM
heatmap_10D(X_df_2_C, gmm_model_2_C, bayesian_model_2_C, title2_C, n_samples=X_df_2_C.shape[0])

# Training and visualizing for Sw_20_T500_Tb_300

In [14]:
# Train the gaussian mixture model and bayesian gaussian mixture model on the following data sets:

title3_C = "Sw_20_T500_Tb_300"
N_G_gmm = 600  # Number of components for GMM
N_G_bayesian = 400  # Number of components for Bayesian GMM

gmm_model_3_C = initialize_gmm(X_df_3_C, N_G_gmm)
save_gmm_model(gmm_model_3_C, f'gmm_model_{title3_C}.pkl')
bayesian_model_3_C = initialize_bayesian_gmm(X_df_3_C, N_G_bayesian)
save_gmm_model(bayesian_model_3_C, f'bayesian_model_{title3_C}.pkl')

c:\Users\jespe\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
c:\Users\jespe\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(


In [ ]:
title3_C = "Sw_20_T500_Tb_300"
gmm_model_3_C = load_gmm_model(f'gmm_model_{title3_C}.pkl')
bayesian_model_3_C = load_gmm_model(f'bayesian_model_{title3_C}.pkl')

# Seperate the visualization and training part
visualize_pdf_10D(X_df_3_C, gmm_model_3_C, bayesian_model_3_C, title3_C, N_G_gmm, N_G_bayesian)
# Plot heatmap of the GMM
heatmap_10D(X_df_3_C, gmm_model_3_C, bayesian_model_3_C, title3_C, n_samples=X_df_3_C.shape[0])

# Postprocessing functionalities

### Inverse liao transform

In [15]:
# Parameters uses for inverse liao transform

def retrieve_X_MD(path_to_data):
    """
    @brief: function that retrieves the unmodified molecular dynamics such that it can be used in the inv. liao transform
    @param path_to_data: path to the file in which this file is located
    @return: X_MD
    """
    X_MD = []
    y_MD = []
    for file in os.listdir(path_to_data):
        if file.endswith(".txt") and "MD" in file:
            X_MD.append(os.path.join(path_to_data,file))
            y_MD.append('')
    
    return X_MD
            

def return_liao_R_transform_parameters(vel):
    """
    @brief: here, input X_MD to extract the necessary parameters to perform the inverse Liao transform
    @param vel: this must be X_MD
    @return: kB -> float, conv_v -> float, theta_in -> float, theta_out -> float
    """
    kB=1.38064852e-23
    conv_v=1.0e2; #convert [An/ps] to [m/s]
    av_num=6.022e23
    mass = 2.0158 * 0.001 / av_num
    mass_kg = mass * 0.001 / av_num
    
    # parameter necessary for normalizing
    v_mp = np.sqrt(2 * kB * wall_temp / mass_kg)
    
    v_mean_in=np.mean(((np.linalg.norm(vel[:,0:3],axis=1).reshape(-1,1))*conv_v)**2)
    v_mean_out=np.mean(((np.linalg.norm(vel[:,3:6],axis=1).reshape(-1,1))*conv_v)**2)
    T_in=v_mean_in*(mass*0.001/6.023e23)/4/kB
    T_out=v_mean_out*(mass*0.001/6.023e23)/4/kB
    theta_in=kB*T_in/(mass*0.001/6.023e23)
    theta_out=kB*T_out/(mass*0.001/6.023e23)
    
    return kB, conv_v, theta_in, theta_out

    


def liao_R_transform(vel,theta_in: float,theta_out: float, wall_temp: int):
    """Function that implements Liao R-transfer function on results"""
    
    # Constants
    conv_v=1.0e2; #convert [An/ps] to [m/s]
    kB=1.38064852e-23
    av_num=6.022e23
    #mass_kg = 2.0158 * 0.001 / av_num
    mass =2* getattr(periodictable,'H').mass
    l_b=0.741e-10
    mass_kg = mass * 0.001 / av_num
    I=(mass_kg/4)*l_b**2
    
    print("old y velocity is given by in: ", vel.iloc[:,1], "and out: ", vel.iloc[:,2])
    
    v_y_in_RTF = (np.sqrt(-2.0*theta_in*np.log(0.5-0.5*erf(
        (vel.iloc[:,1]*conv_v)/(np.sqrt(2*theta_in))
    ))))/conv_v
    
    v_y_out_RTF = (np.sqrt(-2.0*theta_out*np.log(0.5-0.5*erf(
        (vel.iloc[:,4]*conv_v)/(np.sqrt(2*theta_out))
    ))))/conv_v
    
    # parameter necessary for normalizing
    v_mp = np.sqrt(2 * kB * wall_temp / mass_kg)

    
    #vel2 = np.copy(vel)
    vel2 = vel.copy()
    vel2.iloc[:,1] = v_y_in_RTF *conv_v/v_mp # Normalize
    vel2.iloc[:,4] = v_y_out_RTF *conv_v/v_mp # Normalize
    print("new y velocity is given by in: ", vel2.iloc[:,1], "and out: ", vel2.iloc[:,4])
    return vel2


### extract non preprocessed tr_vel (X_MD)

In [16]:
def prepare_data_3(path_to_data, path_save_data, system_6D, system_10D, system_omega, wall_temp, Liao_Transfer_Function):
    x_data = []
    y_data = []

    x_omega = []
    y_omega = []

    x_MD = []
    y_MD = []

    for file in os.listdir(path_to_data):

        if file.endswith(".txt") and "MD" in file:
            x_MD.append(os.path.join(path_to_data,file))
            y_MD.append('')

        if file.endswith(".txt") and "omega" in file:
            x_omega.append(os.path.join(path_to_data,file))
            y_omega.append('')
    print("Data files found:",x_MD)


    ######################################################################################
    #------------------------------- Getting translational velocity data
    ####--------------------------------------------------------------------------------------------------
    for x_data_file,y_data_file in zip(x_MD,y_MD):
        

        conf = VelocityData(x_data_file, frames=None)#[0,30000])
        conf.getRep(rep='vxvyvz',nuc=None) #rep options: vxvyvz, vel2norm, vx2,vy2,vz2,vx,vy,vz
        file_name=x_data_file.replace('.','/')
        file_name=file_name.split('/')
        set_name = file_name[1]
        X = conf.X
        y = conf.y
        if "He" in path_to_data:
            mass = getattr(periodictable,'He').mass
            gas_name = 'He'
        elif "Ar" in path_to_data:
            mass = getattr(periodictable,'Ar').mass
            gas_name = 'Ar'
        elif "H2" in path_to_data:
            mass =2* getattr(periodictable,'H').mass
            gas_name = 'H2'
            l_b=0.741e-10
            mass_kg = mass * 0.001 / av_num
            I=(mass_kg/4)*l_b**2
        elif "N2" in path_to_data:
            mass =2* getattr(periodictable,'N').mass
            gas_name = 'N2'
            l_b=1.097e-10
            mass_kg = mass * 0.001 / av_num
            I=(mass_kg/4)*l_b**2
        else:
            print ("Unable to identify impinging atom type...")
            ele = str(input("Enter the impinging atom symbol : "))
            mass = getattr(periodictable,ele).mass
            v_mp = np.sqrt(2 * kB * wall_temp / mass_kg)
            omega_mp = np.sqrt(2 * kB * wall_temp / I)
        print ('Atomic mass for {} molecule is {:1.4f} \n'.format(gas_name,mass))
        n_MD = X.shape[0]
        print('Number of MD tr velocity data points: {} \n'.format(n_MD))
        # Implementing Liao Transfer function on perpendicular velocity component
        v_TF,T_in,T_out,theta_in,theta_out=func_postprocess.liao_transform(X,mass,'y')
        v_RTF=func_postprocess.liao_R_transform(v_TF,theta_in,theta_out,'y')
        MAC_x,MAC_y,MAC_z,EAC_x,EAC_y,EAC_z,EAC_tr=func_postprocess.compute_AC_correlation_method(X,'Ar','y') #
        AC_MD_tr = [MAC_x,MAC_y,MAC_z,EAC_x,EAC_y,EAC_z,EAC_tr]


    ######################################################################################
    #------------------------------- Getting angular velocity data
    ####--------------------------------------------------------------------------------------------------
    if system_omega:
        for x_data_file2,y_data_file2 in zip(x_omega,y_omega):
            #conf2 = OmegaData(x_data_file2, frames=frames)#[0,30000])
            conf2 = OmegaData(x_data_file2, frames=None)#[0,30000])
            conf2.getRep(rep='omega1omega2',nuc=None) #rep options: vxvyvz, vel2norm, vx2,vy2,vz2,vx,vy,vz
            omega_file_name = x_data_file2.replace('.','/')
            omega_file_name = omega_file_name.split('/')
            omega_name = omega_file_name[1]
            print(omega_name)
            X2 = conf2.X2
            y2 = conf2.y2
            print('Number of MD rot velocity data points: {} \n'.format(X2.shape[0]))
            X2_TF=np.copy(X2)
            Y2_TF=np.copy(y2)
            omega_TF=np.vstack((X2_TF,-X2_TF))
            y2_TF=np.hstack((Y2_TF,-Y2_TF))   

    ######################################################################################
    #------------------------------- Implementing GM model on 10D data
    ####--------------------------------------------------------------------------------------------------                   

    if system_10D:
        SPEED_THRESHOLD = 45  # m/s
        vel_tr = np.copy(X)

        # Speed filtering
        def compute_speed(X: np.ndarray) -> np.ndarray:
            """Compute the speed from the 3D velocity components."""
            return np.sqrt(X[:, 0]**2 + X[:, 1]**2 + X[:, 2]**2)
        speed_in = compute_speed(vel_tr)
        vel_tr = vel_tr[speed_in > SPEED_THRESHOLD]
        X2 = X2[speed_in > SPEED_THRESHOLD]

        vel_tr[:,1]=np.abs(vel_tr[:,1])
        vel_tr[:,4]=np.abs(vel_tr[:,4])
        data_for_AC_MD = np.concatenate((vel_tr,X2),axis=1)

        
        if Liao_Transfer_Function:
            omega_bond = np.copy(X2)
            v_tr=np.copy(v_TF[:n_MD,:])[speed_in > SPEED_THRESHOLD]

            v_omega = np.copy(X2)
            v_mp = np.sqrt(2 * kB * wall_temp / mass_kg)
            omega_mp = np.sqrt(2 * kB * wall_temp / I)

            #--- Normalizing the translational and rotational velocities
            data_train_10D = np.concatenate((v_tr*conv_v/v_mp,X2*conv_omega/omega_mp),axis=1)

            print('The total number of training (after speed filtering) points is: {} \n'.format(data_train_10D.shape[0]))

        else:
            data_train_10D = data_for_AC_MD
            print('The total number of training points is: {} \n'.format(data_train_10D.shape[0]))
    


    
    return vel_tr, data_train_10D, theta_in, theta_out, data_for_AC_MD

In [17]:
def lines_heat_map(v_in ,v_out, yes_abs = False ):# -> (np.array,np.array,np.array):
    """Function to compute refelcite, diffuse and best least square fit into incoming-outgoing velocities"""
    if yes_abs:
        v_in = np.abs(v_in)
        v_out = np.abs(v_out)
    pc = np.polyfit(v_in,v_out, 1)
    coff1 = pc[0]
    coff2 = pc[1]
    v_min = np.min(v_in)
    v_max = np.max(v_in)
    ref_line = 2*np.linspace(2*v_min,2*v_max,100)
    dif_line = np.mean(v_out) * np.ones((len(ref_line)))
    y_fit = coff1 * ref_line+coff2
    return ref_line, dif_line, y_fit, coff1, coff2

### Improved heatmap function with accomodadtion plot

In [69]:
def heatmap_10D_AC_included(vel_tr, X_df, gmm_model, bayesian_model, title, path_to_data, wall_temp, theta_in, theta_out, X_df_3C_non_transformed, n_samples=None):
    """
    Function: heatmap_10D_AC_included
    Visualizes the heatmap of the features in the dataset using both GMM and Bayesian GMM models and adds a line for the EAC.
    @param X_df: DataFrame containing the features.
    @param gmm_model: Fitted Gaussian Mixture Model.
    @param bayesian_model: Fitted Bayesian Gaussian Mixture Model.
    @param title (string): Title for the plot.
    @param path_to_data: path where X_MD can be found (pre-Liao transform)
    """
    if n_samples is None:
        n_samples = X_df.shape[0] # Use the number of rows in X_df if n_samples is not provided

    fntsz = 20
    fig, ax = plt.subplots(5, 3, figsize=(30, 30))
    labels = ["vx_out", "vy_out", "vz_out", "Omega_1_out", "Omega_2_out"]

    # generate samples from the GMM and Bayesian GMM models
    pred_gmm, _ = gmm_model.sample(n_samples)
    print("predictions", pred_gmm.shape)
    pred_bayesian, _ = bayesian_model.sample(n_samples)
    
    assert not np.isinf(pred_gmm).any(), "Infs in pred GMM data"
    
    kB,conv_v,conv_omega,av_num = 1.38064852e-23, 1.0e2, 1.0e12, 6.022e23
    mass = 2* getattr(periodictable,'H').mass
    mass_kg = mass * 0.001 / av_num
    l_b=0.741e-10
    I=(mass_kg/4)*l_b**2
    
    # parameter necessary for normalizing
    v_mp = np.sqrt(2 * kB * wall_temp / mass_kg)
    omega_mp = np.sqrt(2 * kB * wall_temp / I)
    
    
    vel_pred_gmm = pred_gmm[0]
    vel_pred_bayesian = pred_bayesian[0]
    
    #v_pred_gmm=np.copy(pred_gmm[0])
    #v_pred_bayesian=np.copy(pred_bayesian[0])
    v_pred_gmm=np.copy(pred_gmm)
    v_pred_bayesian=np.copy(pred_bayesian)
    
    v_pred_gmm[:,:6] = (v_pred_gmm[:,:6]*v_mp)/conv_v
    v_pred_bayesian[:,:6] = (v_pred_bayesian[:,:6]*v_mp)/conv_v
    
    #assert not np.isinf(v_pred_gmm).any(), "Infs in v_pred_gmm after division by conv_V"
    
    v_pred_gmm[:,6:] = (v_pred_gmm[:,6:]*omega_mp)/conv_omega
    v_pred_bayesian[:,6:] = (v_pred_bayesian[:,6:]*omega_mp)/conv_omega
    
    v_RTF_pred_gmm=func_postprocess.liao_R_transform(v_pred_gmm[:,:6],theta_in,theta_out,'y') # using RTF on predicted results
    v_RTF_pred_bayesian=func_postprocess.liao_R_transform(v_pred_bayesian[:,:6],theta_in,theta_out,'y') # using RTF on predicted results
    
    #assert not np.isinf(v_RTF_pred_gmm).any(), "Infs in v_pred_gmm after inv. liao transform"
    
    v_RTF_pred_gmm = np.append(v_RTF_pred_gmm,v_pred_gmm[:,6:],axis=1)
    v_RTF_pred_bayesian = np.append(v_RTF_pred_bayesian,v_pred_bayesian[:,6:],axis=1)
    
    #v_RTF_pred_gmm = np.concatenate((v_RTF_pred_gmm[:,:6]*conv_v/v_mp,v_RTF_pred_gmm[:,6:]*conv_omega/omega_mp),axis=1)
    #v_RTF_pred_bayesian = np.concatenate((v_RTF_pred_gmm[:,:6]*conv_v/v_mp,v_RTF_pred_gmm[:,6:]*conv_omega/omega_mp),axis=1)
    
    #print(v_RTF_pred_gmm)

    # After creating v_RTF_pred_gmm
    #assert not np.isnan(v_RTF_pred_gmm).any(), "NaNs in transformed GMM data"
    #assert not np.isinf(v_RTF_pred_gmm).any(), "Infs in transformed GMM data"
    
    # Normalize v_RTF_pred_gmm, v_RTF_pred_bayesian and X_df_3c_non_transformed
    # Convert MD reference data to NumPy if it's not already
    #X_md = np.asarray(X_df_3C_non_transformed)
    #means = np.mean(X_md, axis=0)
    #stds = np.std(X_md, axis=0)
    #stds[stds == 0] = 1.0  # Prevent division by zero

    # Apply normalization
    #X_df_3C_non_transformed = (X_md - means) / stds
    #v_RTF_pred_gmm = (v_RTF_pred_gmm - means) / stds
    #v_RTF_pred_bayesian = (v_RTF_pred_bayesian - means) / stds

    
    # Store AC's
    MAC_x,MAC_y,MAC_z,EAC_x,EAC_y,EAC_z,EAC_tr,EAC_tr_energy,EAC_rot,EAC_rot_energy,EAC_tot_energy = func_postprocess.compute_AC_correlation_method(X_df_3C_non_transformed, 'H2', 'y')
    AC_MD = [MAC_x,MAC_y,MAC_z,EAC_x,EAC_y,EAC_z,EAC_tr,EAC_tr_energy,EAC_rot,EAC_rot_energy,EAC_tot_energy]
    MAC_x,MAC_y,MAC_z,EAC_x,EAC_y,EAC_z,EAC_tr,EAC_tr_energy,EAC_rot,EAC_rot_energy,EAC_tot_energy = func_postprocess.compute_AC_correlation_method(v_RTF_pred_gmm, 'H2', 'y')
    AC_GMM = [MAC_x,MAC_y,MAC_z,EAC_x,EAC_y,EAC_z,EAC_tr,EAC_tr_energy,EAC_rot,EAC_rot_energy,EAC_tot_energy]
    MAC_x,MAC_y,MAC_z,EAC_x,EAC_y,EAC_z,EAC_tr,EAC_tr_energy,EAC_rot,EAC_rot_energy,EAC_tot_energy = func_postprocess.compute_AC_correlation_method(v_RTF_pred_bayesian, 'H2', 'y')
    AC_bayesian = [MAC_x,MAC_y,MAC_z,EAC_x,EAC_y,EAC_z,EAC_tr,EAC_tr_energy,EAC_rot,EAC_rot_energy,EAC_tot_energy]
    # Store into dataframe
    ACs = [AC_MD, AC_GMM, AC_bayesian]
    df_ACs = pd.DataFrame(ACs, index=['MD', 'GMM', 'Bayesian'], columns=['MAC_x','MAC_y','MAC_z','EAC_x','EAC_y','EAC_z','EAC_tr','EAC_tr_energy','EAC_rot','EAC_rot_energy','EAC_tot_energy'])

    


    #---------------------------------------------------------------------------------------------

    
    #---------------------------------------------------------------------------------------------
    
    # Convert prepared samples to a DF
    samples_gmm_df_RF = pd.DataFrame(v_RTF_pred_gmm, columns=['vx_in', 'vy_in', 'vz_in', 'vx_out', 'vy_out', 'vz_out', 'omega_1_in', 'omega_2_in', 'omega_1_out', 'omega_2_out'], copy=True)
    samples_bayesian_df_RF = pd.DataFrame(v_RTF_pred_bayesian, columns=['vx_in', 'vy_in', 'vz_in', 'vx_out', 'vy_out', 'vz_out', 'omega_1_in', 'omega_2_in', 'omega_1_out', 'omega_2_out'], copy=True)
    R_X_df = pd.DataFrame(X_df_3C_non_transformed, columns=['vx_in', 'vy_in', 'vz_in', 'vx_out', 'vy_out', 'vz_out', 'omega_1_in', 'omega_2_in', 'omega_1_out', 'omega_2_out'], copy=True)
    #print(samples_bayesian_df_RF.head())
    
    # Least square linear fit of the data

    in_index = [0, 1, 2, 6, 7]  # Indices for the input features
    out_index = [3, 4, 5, 8, 9]  # Indices for the output features
    coefficients = []
    coff_1_title = "slope"
    coff_2_title = "intercept"

    # Loop through the features to create subplots
    for r in range(5):
        # Compute density for MD data
        #xy_MD = np.vstack([X_df.iloc[:n_samples, in_index[r]], X_df.iloc[:n_samples, out_index[r]]])
        xy_MD = np.vstack([R_X_df.iloc[:n_samples, in_index[r]], R_X_df.iloc[:n_samples, out_index[r]]])
        density_MD = gaussian_kde(xy_MD)(xy_MD)
        
        # Least square fit of the data
        #ref_line_MD, dif_line_MD, y_fit_MD = lines_heat_map(X_df.iloc[:n_samples, in_index[r]] ,X_df.iloc[:n_samples, out_index[r]])
        ref_line_MD, dif_line_MD, y_fit_MD, coff_1, coff_2 = lines_heat_map(R_X_df.iloc[:n_samples, in_index[r]] ,R_X_df.iloc[:n_samples, out_index[r]])
        
        #print(ref_line_MD)

        # plot the MD data
        #ax[r, 0].scatter(X_df.iloc[:n_samples, in_index[r]], X_df.iloc[:n_samples, out_index[r]], c=density_MD, s=15, cmap='jet', alpha=0.6)
        ax[r, 0].scatter(R_X_df.iloc[:n_samples, in_index[r]], R_X_df.iloc[:n_samples, out_index[r]], c=density_MD, s=15, cmap='jet', alpha=0.6)
        ax[r,0].plot(ref_line_MD, dif_line_MD)
        ax[r,0].plot(ref_line_MD, y_fit_MD)
        # Write down coff_1 and coff_2 (slope and intercept)
        textstr = f"{coff_1_title}: {coff_1:.2f}\n{coff_2_title}: {coff_2:.2f}"
        ax[r, 0].text(0.05, 0.95, textstr, transform=ax[r, 0].transAxes,fontsize=14, verticalalignment='top', bbox=dict(boxstyle="round", facecolor='white', alpha=0.7))

        # Compute density for GMM samples
        xy_GMM = np.vstack([samples_gmm_df_RF.iloc[:n_samples, in_index[r]], samples_gmm_df_RF.iloc[:n_samples, out_index[r]]])
        density_GMM = gaussian_kde(xy_GMM)(xy_GMM)
        
        # Least square fit of the data
        ref_line_GMM, dif_line_GMM, y_fit_GMM, coff_1, coff_2 = lines_heat_map(samples_gmm_df_RF.iloc[:n_samples, in_index[r]] ,samples_gmm_df_RF.iloc[:n_samples, out_index[r]])
        coefficients.append(coff_1)
        coefficients.append(coff_2)

        # plot the GMM data
        ax[r, 1].scatter(samples_gmm_df_RF.iloc[:n_samples, in_index[r]], samples_gmm_df_RF.iloc[:n_samples, out_index[r]], c=density_GMM, s=15, cmap='jet', alpha=0.6)
        ax[r,1].plot(ref_line_GMM, dif_line_GMM)
        ax[r,1].plot(ref_line_GMM, y_fit_GMM)
        # Write down coff_1 and coff_2 (slope and intercept)
        textstr = f"{coff_1_title}: {coff_1:.2f}\n{coff_2_title}: {coff_2:.2f}"
        ax[r, 1].text(0.05, 0.95, textstr, transform=ax[r, 1].transAxes,fontsize=14, verticalalignment='top', bbox=dict(boxstyle="round", facecolor='white', alpha=0.7))


        # Compute density for Bayesian samples
        xy_Bayesian = np.vstack([samples_bayesian_df_RF.iloc[:n_samples, in_index[r]], samples_bayesian_df_RF.iloc[:n_samples, out_index[r]]])
        density_Bayesian = gaussian_kde(xy_Bayesian)(xy_Bayesian)
        
        # Least square fit of the data
        ref_line_bayesian, dif_line_bayesian, y_fit_bayesian, coff_1, coff_2 = lines_heat_map(samples_bayesian_df_RF.iloc[:n_samples, in_index[r]] ,samples_bayesian_df_RF.iloc[:n_samples, out_index[r]])
        coefficients.append(coff_1)
        coefficients.append(coff_2)

        # plot the Bayesian data
        ax[r, 2].scatter(samples_bayesian_df_RF.iloc[:n_samples, in_index[r]], samples_bayesian_df_RF.iloc[:n_samples, out_index[r]], c=density_Bayesian, s=15, cmap='jet', alpha=0.6)
        ax[r,2].plot(ref_line_bayesian, dif_line_bayesian)
        ax[r,2].plot(ref_line_bayesian, y_fit_bayesian)
        # Write down coff_1 and coff_2 (slope and intercept)
        textstr = f"{coff_1_title}: {coff_1:.2f}\n{coff_2_title}: {coff_2:.2f}"
        ax[r, 2].text(0.05, 0.95, textstr, transform=ax[r, 2].transAxes,fontsize=14, verticalalignment='top', bbox=dict(boxstyle="round", facecolor='white', alpha=0.7))

    # row labels
    ax[0, 0].set_ylabel('vx', fontsize=fntsz)
    ax[1, 0].set_ylabel('vy', fontsize=fntsz)
    ax[2, 0].set_ylabel('vz', fontsize=fntsz)
    ax[3, 0].set_ylabel('Omega_1', fontsize=fntsz)
    ax[4, 0].set_ylabel('Omega_2', fontsize=fntsz)
    # column labels
    ax[0, 0].set_title("MD", fontsize=fntsz)
    ax[0, 1].set_title("GMM", fontsize=fntsz)
    ax[0, 2].set_title("Bayesian GMM", fontsize=fntsz)
    
    
    
    for row_idx, row in enumerate(ax):
        for a in row:
            a.set_xlim(-50, 50)
            a.set_ylim(-50, 50)
            if row_idx == 1:  # Second row (index 1)
                a.set_xlim(0, 600)  # New xlim for second row
                a.set_ylim(0, 600)  # New xlim for second row



    fig.suptitle("heatmap "+ title, fontsize=24)
    fig.savefig("heatmap_and_least_square_fit_"+ title + '.png', dpi=300, bbox_inches='tight')
    
    return df_ACs


### Test the new heatmap function

In [19]:
er_type = 'MAE'

system_6D = False 
system_10D = True # Assignment C uses 10D data since we now take rotational velocity data into account
system_omega = True 
Liao_Transfer_Function = True

kB,conv_v,conv_omega,av_num = 1.38064852e-23,1.0e2,1.0e12,6.022e23

In [50]:
"""
H2_omega_Sw_20_T500_Tb_300.txt and H2_Vel_MD_Sw_20_T500_Tb_300.txt
Bottom wall temperature = 300 K
Top wall temperature = 500 K
"""
path_to_data = 'Data_H2_assignment_bayesian_C_3'
path_save_data='./'+path_to_data
wall_temp = 300 # Wall temperature in Kelvin
vel_tr, X_df_3C, theta_in, theta_out, X_df_3C_non_transformed = prepare_data_3(path_to_data, path_save_data, system_6D, system_10D, system_omega, wall_temp, Liao_Transfer_Function)[:3000]
# TODO: Liao transorm
X_df_3C = pd.DataFrame(X_df_3C, columns=['vx_in', 'vy_in', 'vz_in', 'vx_out', 'vy_out', 'vz_out', 'omega_1_in', 'omega_2_in', 'omega_1_out', 'omega_2_out'], copy=True)# Extract X_df with inverse liao transform performed
type(vel_tr)

Data files found: ['Data_H2_assignment_bayesian_C_3\\H2_Vel_MD_Sw_20_T500_Tb_300.txt']
Generating Representation


0% [#####                         ] 100% | ETA: 00:00:00

Not a float


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:01


Atomic mass for H2 molecule is 2.0159 

Number of MD tr velocity data points: 184022 

Generating Representation


0% [#########                     ] 100% | ETA: 00:00:00

Not a float


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


txt
Number of MD rot velocity data points: 184022 

The total number of training (after speed filtering) points is: 5729 



numpy.ndarray

In [51]:
# Train the gaussian mixture model and bayesian gaussian mixture model on the following data sets:

title3_C = "Sw_20_T500_Tb_300"
N_G_gmm = 600  # Number of components for GMM
N_G_bayesian = 400  # Number of components for Bayesian GMM

gmm_model_3_C = initialize_gmm(X_df_3_C, N_G_gmm)
save_gmm_model(gmm_model_3_C, f'gmm_model_{title3_C}.pkl')
bayesian_model_3_C = initialize_bayesian_gmm(X_df_3_C, N_G_bayesian)
save_gmm_model(bayesian_model_3_C, f'bayesian_model_{title3_C}.pkl')

c:\Users\jespe\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
c:\Users\jespe\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(


In [70]:
title3_C = "Sw_20_T500_Tb_300"
gmm_model_3_C = load_gmm_model(f'gmm_model_{title3_C}.pkl')
bayesian_model_3_C = load_gmm_model(f'bayesian_model_{title3_C}.pkl')
N_G_gmm = 600  # Number of components for GMM
N_G_bayesian = 400  # Number of components for Bayesian GMM

# Seperate the visualization and training part
visualize_pdf_10D(X_df_3_C, gmm_model_3_C, bayesian_model_3_C, title3_C, N_G_gmm, N_G_bayesian)

print(theta_in, theta_out, "theta in and out")
# Plot heatmap of the GMM
df_AC = heatmap_10D_AC_included(vel_tr, X_df_3_C, gmm_model_3_C, bayesian_model_3_C, title3_C, path_to_data, wall_temp, theta_in, theta_out, X_df_3C_non_transformed, n_samples=X_df_3_C.shape[0])



C:\Users\jespe\AppData\Local\Temp\ipykernel_34360\2729003733.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


1861213.6702574228 1948812.9363073888 theta in and out
predictions (3000, 10)


In [71]:
print(df_AC)

             MAC_x     MAC_y     MAC_z     EAC_x     EAC_y     EAC_z  \
MD        0.780767  0.926143  0.766497  0.880367  0.912603  0.874972   
GMM       0.820995  0.918499  0.771986  0.865662  0.910792  0.868395   
Bayesian  0.784133  0.912468  0.773038  0.815608  0.887194  0.819092   

            EAC_tr  EAC_tr_energy   EAC_rot  EAC_rot_energy  EAC_tot_energy  
MD        0.255423       0.255423  0.504406        0.504406        0.082802  
GMM       0.299300       0.299300  0.547796        0.547796        0.194479  
Bayesian  0.503766       0.503766  0.466363        0.466363        0.341730  
